# A/B Testing Lifecycle: Friction Warning Experiment

## Business Context

A short-video platform is evaluating a **friction warning** intervention to reduce policy-violating content. When a lightweight classifier flags a post as potentially violating community guidelines, treatment users see an interstitial warning ("This may violate community guidelines -- are you sure?") before their post is published. Control users post without interruption.

This notebook walks through the **complete A/B testing lifecycle** for this experiment:

1. **Pre-experiment**: Power analysis, sample size calculation, covariate balance checks
2. **During experiment**: Sequential monitoring, SRM detection, alpha spending
3. **Post-experiment**: Primary analysis (naive and CUPED), subgroup heterogeneity, guardrail checks
4. **Decision**: Launch recommendation with cost-benefit analysis and post-launch monitoring

## Learning Objectives

- Design and validate an A/B test end-to-end for a content safety intervention
- Apply CUPED variance reduction using pre-experiment covariates
- Implement sequential testing with mSPRT and O'Brien-Fleming spending functions
- Conduct subgroup analysis with forest plots
- Evaluate guardrail metrics and formulate a launch recommendation
- Detect novelty effects in post-experiment monitoring

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly not available; falling back to matplotlib for all plots.")

from data.generators.friction_experiment import generate_friction_experiment
from utils.statistical_tests import (
    two_sample_z_test,
    proportion_z_test,
    cuped_adjustment,
    bootstrap_ci,
    sample_size_proportion,
    sequential_test,
    chi_squared_test,
    multiple_testing_correction,
)
from utils.visualization import set_style, COLORS, plot_ab_test_results, plot_sequential_test

set_style()
np.random.seed(42)
%matplotlib inline

print("Setup complete.")

## Business Context: The Friction Warning Intervention

### Intervention Design

When a user submits content, a lightweight classifier scores it in real time. If the score exceeds a threshold (roughly the top 12% of posts), the user sees an interstitial warning:

> *"This content may violate community guidelines. Are you sure you want to post?"*

The user can either **dismiss** the warning and post anyway, or **abandon** the post. Control users never see this warning.

### Hypothesis

- **H0**: The friction warning has no effect on the violation rate.
- **H1**: The friction warning reduces the post-experiment violation rate by at least 8% relative (from ~4% to ~3.7%).

### Metric Taxonomy

| Role | Metric | Definition |
|------|--------|------------|
| **Primary (OEC)** | Violation rate | `post_violations / post_content_count` per user |
| **Guardrail** | Posting completion rate | `posts_completed / posts_started` |
| **Guardrail** | Creator activity | `days_active_post` in 28-day window |
| **Diagnostic** | Warning dismissal rate | `warning_dismissed / warned_count` (treatment only) |
| **Data quality** | Sample Ratio Mismatch | Chi-squared test on arm sizes |

In [ ]:
# Generate the experiment data
df = generate_friction_experiment(n_users=50_000, seed=42)
print(f"Dataset shape: {df.shape[0]:,} users x {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nTreatment distribution:")
print(df["treatment"].value_counts().rename({0: "control", 1: "treatment"}))
df.head()

In [ ]:
# ============================================================
# Exploratory Data Analysis
# ============================================================

ctrl = df[df.treatment == 0]
treat = df[df.treatment == 1]
n_ctrl, n_treat = len(ctrl), len(treat)

print("=== Sample Sizes ===")
print(f"Control:   {n_ctrl:>7,}")
print(f"Treatment: {n_treat:>7,}")
print(f"Total:     {n_ctrl + n_treat:>7,}")
print(f"Ratio (T/C): {n_treat / n_ctrl:.4f}")

# --- Sample Ratio Mismatch (SRM) ---
expected_ratio = 1.0
total = n_ctrl + n_treat
expected_ctrl = total / (1 + expected_ratio)
expected_treat = total * expected_ratio / (1 + expected_ratio)
chi2_srm = ((n_ctrl - expected_ctrl)**2 / expected_ctrl +
            (n_treat - expected_treat)**2 / expected_treat)
p_srm = 1 - stats.chi2.cdf(chi2_srm, df=1)
print(f"\n=== Sample Ratio Mismatch Test ===")
print(f"Chi2 = {chi2_srm:.4f}, p = {p_srm:.4f}")
print(f"SRM detected: {'YES' if p_srm < 0.001 else 'No'}")

# --- Covariate balance ---
print("\n=== Covariate Balance ===")
continuous_covariates = ["pre_violation_rate", "account_age_days", "follower_count"]
print(f"{'Covariate':<25} {'Control Mean':>14} {'Treatment Mean':>16} {'Diff':>10} {'p-value':>10}")
print("-" * 78)
for var in continuous_covariates:
    c_mean = ctrl[var].mean()
    t_mean = treat[var].mean()
    t_stat, p_val = stats.ttest_ind(ctrl[var], treat[var])
    diff = t_mean - c_mean
    flag = " *" if p_val < 0.05 else ""
    print(f"{var:<25} {c_mean:>14.5f} {t_mean:>16.5f} {diff:>10.5f} {p_val:>10.4f}{flag}")

# Region distribution balance
print("\n=== Region Distribution ===")
region_ctrl = ctrl["region"].value_counts(normalize=True).sort_index()
region_treat = treat["region"].value_counts(normalize=True).sort_index()
region_df = pd.DataFrame({"Control": region_ctrl, "Treatment": region_treat})
region_df["Diff"] = region_df["Treatment"] - region_df["Control"]
print(region_df.round(4).to_string())

# Chi-squared test on region distribution
region_ct = pd.crosstab(df["treatment"], df["region"])
chi2_reg, p_reg, _, _ = stats.chi2_contingency(region_ct.values)
print(f"\nRegion balance chi2 = {chi2_reg:.4f}, p = {p_reg:.4f}")

# Content type distribution
print("\n=== Content Type Distribution ===")
ct_ctrl = ctrl["content_type_primary"].value_counts(normalize=True).sort_index()
ct_treat = treat["content_type_primary"].value_counts(normalize=True).sort_index()
ct_df = pd.DataFrame({"Control": ct_ctrl, "Treatment": ct_treat})
ct_df["Diff"] = ct_df["Treatment"] - ct_df["Control"]
print(ct_df.round(4).to_string())

In [ ]:
# ============================================================
# Power Analysis
# ============================================================

# Baseline violation rate from control arm
baseline_vr = ctrl["post_violation_rate"].mean()
print(f"Baseline violation rate (control): {baseline_vr:.5f}")

# MDE: 8% relative reduction
relative_mde = 0.08
absolute_mde = baseline_vr * relative_mde
print(f"Target relative MDE: {relative_mde:.0%}")
print(f"Target absolute MDE: {absolute_mde:.5f}")

# Required sample size
n_required_80 = sample_size_proportion(baseline_vr, absolute_mde, alpha=0.05, power=0.80)
n_required_90 = sample_size_proportion(baseline_vr, absolute_mde, alpha=0.05, power=0.90)

print(f"\n=== Required Sample Size per Arm ===")
print(f"80% power: {n_required_80:>10,}")
print(f"90% power: {n_required_90:>10,}")
print(f"Available:  {n_ctrl:>10,} per arm")
print(f"Sufficient for 80% power: {'Yes' if n_ctrl >= n_required_80 else 'No'}")
print(f"Sufficient for 90% power: {'Yes' if n_ctrl >= n_required_90 else 'No'}")

# --- Power curves at different effect sizes ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Sample size vs MDE
relative_mdes = np.arange(0.03, 0.20, 0.005)
for power_level, color, ls in [(0.80, COLORS["control"], "-"), (0.90, COLORS["treatment"], "--")]:
    sample_sizes = []
    for rmde in relative_mdes:
        abs_mde = baseline_vr * rmde
        n = sample_size_proportion(baseline_vr, abs_mde, alpha=0.05, power=power_level)
        sample_sizes.append(n)
    axes[0].plot(relative_mdes * 100, sample_sizes, color=color, linestyle=ls,
                 linewidth=2, label=f"Power = {power_level:.0%}")

axes[0].axhline(n_ctrl, color="gray", linestyle=":", linewidth=1.5, label=f"Available n = {n_ctrl:,}")
axes[0].axvline(relative_mde * 100, color=COLORS["danger"], linestyle=":", alpha=0.7,
                label=f"Target MDE = {relative_mde:.0%}")
axes[0].set_xlabel("Relative MDE (%)")
axes[0].set_ylabel("Required Sample Size per Arm")
axes[0].set_title("Sample Size vs Minimum Detectable Effect")
axes[0].set_yscale("log")
axes[0].legend(fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f"{x:,.0f}"))

# Right: Power curve at our sample size
rmde_range = np.arange(0.02, 0.25, 0.005)
powers = []
for rmde in rmde_range:
    abs_mde = baseline_vr * rmde
    se = np.sqrt(2 * baseline_vr * (1 - baseline_vr) / n_ctrl)
    z_alpha = stats.norm.ppf(0.975)
    z_power = abs_mde / se - z_alpha
    pwr = stats.norm.cdf(z_power)
    powers.append(pwr)

axes[1].plot(rmde_range * 100, powers, color=COLORS["treatment"], linewidth=2.5)
axes[1].axhline(0.80, color="gray", linestyle="--", alpha=0.5, label="80% power")
axes[1].axhline(0.90, color="gray", linestyle=":", alpha=0.5, label="90% power")
axes[1].axvline(relative_mde * 100, color=COLORS["danger"], linestyle=":", alpha=0.7,
                label=f"Target MDE = {relative_mde:.0%}")
axes[1].set_xlabel("Relative MDE (%)")
axes[1].set_ylabel("Statistical Power")
axes[1].set_title(f"Power Curve (n = {n_ctrl:,} per arm)")
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# Print the power at our target MDE
se_target = np.sqrt(2 * baseline_vr * (1 - baseline_vr) / n_ctrl)
z_power_target = absolute_mde / se_target - stats.norm.ppf(0.975)
power_at_target = stats.norm.cdf(z_power_target)
print(f"\nPower at target MDE ({relative_mde:.0%} relative): {power_at_target:.2%}")

In [ ]:
# ============================================================
# Pre-Experiment Validation
# ============================================================

print("=== AA Test: Pre-Existing Differences ===")
print("Testing that treatment and control are identical on pre-experiment metric.\n")

# AA test on pre_violation_rate (the CUPED covariate)
aa_test = two_sample_z_test(
    ctrl["pre_violation_rate"].values,
    treat["pre_violation_rate"].values
)
print(f"Pre-violation rate:")
print(f"  Control mean:   {aa_test['control_mean']:.6f}")
print(f"  Treatment mean: {aa_test['treatment_mean']:.6f}")
print(f"  Difference:     {aa_test['effect_size']:.6f}")
print(f"  Z-stat:         {aa_test['z_stat']:.4f}")
print(f"  p-value:        {aa_test['p_value']:.4f}")
print(f"  Conclusion:     {'FAIL - pre-existing difference detected' if aa_test['p_value'] < 0.05 else 'PASS - no pre-existing difference'}")

# Covariate balance: continuous variables
print("\n=== Covariate Balance: Continuous Variables ===")
balance_results = []
for var in ["pre_violation_rate", "account_age_days", "follower_count"]:
    t_stat, p_val = stats.ttest_ind(ctrl[var], treat[var])
    pooled_std = np.sqrt((ctrl[var].var() + treat[var].var()) / 2)
    smd = abs(treat[var].mean() - ctrl[var].mean()) / pooled_std if pooled_std > 0 else 0
    balance_results.append({
        "covariate": var,
        "t_stat": t_stat,
        "p_value": p_val,
        "smd": smd,
        "pass": smd < 0.1
    })

balance_df = pd.DataFrame(balance_results)
print(balance_df.to_string(index=False, float_format="%.4f"))

# Covariate balance: categorical variables
print("\n=== Covariate Balance: Categorical Variables ===")
for cat_var in ["region", "content_type_primary"]:
    ct = pd.crosstab(df["treatment"], df[cat_var])
    chi2_res = chi_squared_test(ct.values)
    print(f"{cat_var}: chi2 = {chi2_res['chi2']:.4f}, p = {chi2_res['p_value']:.4f}, "
          f"Cramer's V = {chi2_res['cramers_v']:.4f} "
          f"[{'PASS' if chi2_res['p_value'] > 0.05 else 'FAIL'}]")

# SRM test
print("\n=== Sample Ratio Mismatch ===")
srm_result = proportion_z_test(
    n_ctrl, n_ctrl + n_treat,
    n_treat, n_ctrl + n_treat,
    alternative="two-sided"
)
print(f"Control: {n_ctrl:,}, Treatment: {n_treat:,}")
print(f"Ratio: {n_treat / n_ctrl:.6f}")
print(f"SRM p-value: {p_srm:.4f}")
print(f"SRM detected (p < 0.001): {'YES' if p_srm < 0.001 else 'No'}")

print("\n" + "=" * 50)
all_pass = (aa_test['p_value'] > 0.05 and
            all(r['pass'] for r in balance_results) and
            p_srm > 0.001)
print(f"PRE-EXPERIMENT VALIDATION: {'ALL CHECKS PASSED' if all_pass else 'ISSUES DETECTED'}")

In [ ]:
# ============================================================
# Primary Analysis -- Naive (Simple Z-test)
# ============================================================

ctrl_vr = ctrl["post_violation_rate"].values
treat_vr = treat["post_violation_rate"].values

naive_result = two_sample_z_test(ctrl_vr, treat_vr)

print("=" * 55)
print("PRIMARY ANALYSIS: Post-Experiment Violation Rate")
print("=" * 55)
print(f"\nControl mean:     {naive_result['control_mean']:.6f}")
print(f"Treatment mean:   {naive_result['treatment_mean']:.6f}")
print(f"Absolute effect:  {naive_result['effect_size']:.6f}")
print(f"Relative effect:  {naive_result['effect_size'] / naive_result['control_mean']:.2%}")
print(f"\nZ-statistic:      {naive_result['z_stat']:.4f}")
print(f"p-value:          {naive_result['p_value']:.6f}")
print(f"Cohen's d:        {naive_result['cohens_d']:.4f}")
print(f"95% CI:           [{naive_result['ci_lower']:.6f}, {naive_result['ci_upper']:.6f}]")
print(f"CI width:         {naive_result['ci_upper'] - naive_result['ci_lower']:.6f}")
print(f"\nStatistically significant (p < 0.05): {'Yes' if naive_result['p_value'] < 0.05 else 'No'}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Distribution comparison
axes[0].hist(ctrl_vr, bins=60, alpha=0.6, color=COLORS["control"],
             label=f"Control (n={n_ctrl:,})", density=True)
axes[0].hist(treat_vr, bins=60, alpha=0.6, color=COLORS["treatment"],
             label=f"Treatment (n={n_treat:,})", density=True)
axes[0].axvline(np.mean(ctrl_vr), color=COLORS["control"], linestyle="--", linewidth=2)
axes[0].axvline(np.mean(treat_vr), color=COLORS["treatment"], linestyle="--", linewidth=2)
axes[0].set_xlabel("Post Violation Rate")
axes[0].set_ylabel("Density")
axes[0].set_title("Violation Rate Distribution by Group")
axes[0].legend()

# Right: Effect size with CI
effect = naive_result["effect_size"]
ci_lo = naive_result["ci_lower"]
ci_hi = naive_result["ci_upper"]
sig_color = COLORS["accent"] if naive_result["p_value"] < 0.05 else COLORS["neutral"]

axes[1].barh(0, effect, color=sig_color, alpha=0.7, height=0.3)
axes[1].errorbar(effect, 0, xerr=[[effect - ci_lo], [ci_hi - effect]],
                 fmt="o", color="black", capsize=8, markersize=10)
axes[1].axvline(0, color="gray", linestyle="-", linewidth=1)
axes[1].set_xlabel("Treatment Effect (Violation Rate)")
axes[1].set_title(f"Naive Z-test: Effect = {effect:.5f} (p = {naive_result['p_value']:.4f})")
axes[1].set_yticks([])
axes[1].text(effect, 0.25, f"95% CI: [{ci_lo:.5f}, {ci_hi:.5f}]",
             ha="center", fontsize=10)

plt.tight_layout()
plt.show()

### Interpretation

The naive z-test provides our baseline estimate. The treatment (friction warning) reduces the violation rate, but the confidence interval is relatively wide due to the high variance in individual-level violation rates. In the next section, we use CUPED to sharpen this estimate.

In [ ]:
# ============================================================
# CUPED Variance Reduction
# ============================================================

# Use pre_violation_rate as the CUPED covariate
metric_post = df["post_violation_rate"].values
metric_pre = df["pre_violation_rate"].values
group = df["treatment"].values

# Check correlation between pre and post metrics
corr = np.corrcoef(metric_pre, metric_post)[0, 1]
print(f"Correlation between pre and post violation rate: r = {corr:.4f}")
print(f"Expected variance reduction (r^2): {corr**2:.2%}")

# Apply CUPED
adjusted_metric, variance_reduction = cuped_adjustment(metric_post, metric_pre, group)

print(f"\n=== CUPED Results ===")
print(f"Variance of raw metric:      {np.var(metric_post):.8f}")
print(f"Variance of CUPED metric:    {np.var(adjusted_metric):.8f}")
print(f"Variance reduction:          {variance_reduction:.2%}")
print(f"Effective sample size boost: {1 / (1 - variance_reduction):.1f}x")

# CUPED-adjusted test
ctrl_cuped = adjusted_metric[group == 0]
treat_cuped = adjusted_metric[group == 1]
cuped_result = two_sample_z_test(ctrl_cuped, treat_cuped)

# Comparison table
print(f"\n{'Metric':<25} {'Naive':>14} {'CUPED':>14}")
print("-" * 55)
print(f"{'Control mean':<25} {naive_result['control_mean']:>14.6f} {cuped_result['control_mean']:>14.6f}")
print(f"{'Treatment mean':<25} {naive_result['treatment_mean']:>14.6f} {cuped_result['treatment_mean']:>14.6f}")
print(f"{'Effect size':<25} {naive_result['effect_size']:>14.6f} {cuped_result['effect_size']:>14.6f}")
print(f"{'Z-statistic':<25} {naive_result['z_stat']:>14.4f} {cuped_result['z_stat']:>14.4f}")
print(f"{'p-value':<25} {naive_result['p_value']:>14.6f} {cuped_result['p_value']:>14.6f}")
print(f"{'CI lower':<25} {naive_result['ci_lower']:>14.6f} {cuped_result['ci_lower']:>14.6f}")
print(f"{'CI upper':<25} {naive_result['ci_upper']:>14.6f} {cuped_result['ci_upper']:>14.6f}")

naive_ci_width = naive_result['ci_upper'] - naive_result['ci_lower']
cuped_ci_width = cuped_result['ci_upper'] - cuped_result['ci_lower']
ci_reduction = 1 - cuped_ci_width / naive_ci_width
print(f"{'CI width':<25} {naive_ci_width:>14.6f} {cuped_ci_width:>14.6f}")
print(f"\nCI width reduction: {ci_reduction:.1%}")

# Visualization: CI comparison
fig, ax = plt.subplots(figsize=(10, 4))

methods = ["Naive Z-test", "CUPED-adjusted"]
effects = [naive_result["effect_size"], cuped_result["effect_size"]]
ci_lowers = [naive_result["ci_lower"], cuped_result["ci_lower"]]
ci_uppers = [naive_result["ci_upper"], cuped_result["ci_upper"]]
colors_ci = [COLORS["control"], COLORS["treatment"]]

for i, (method, eff, lo, hi, c) in enumerate(zip(methods, effects, ci_lowers, ci_uppers, colors_ci)):
    ax.errorbar(eff, i, xerr=[[eff - lo], [hi - eff]],
                fmt="o", color=c, capsize=8, markersize=10, linewidth=2,
                label=f"{method}: [{lo:.5f}, {hi:.5f}]")

ax.axvline(0, color="gray", linestyle="-", linewidth=1)
ax.set_yticks([0, 1])
ax.set_yticklabels(methods)
ax.set_xlabel("Treatment Effect (Violation Rate Difference)")
ax.set_title(f"CUPED reduces CI width by {ci_reduction:.0%}")
ax.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

### CUPED Takeaway

CUPED leverages the correlation between the pre-experiment violation rate and the post-experiment violation rate to reduce variance. The stronger the correlation, the greater the variance reduction. This is equivalent to running the experiment with a much larger sample size, making it easier to detect smaller effects.

In [ ]:
# ============================================================
# Sequential Testing
# ============================================================

# Simulate daily accumulation of the experiment
# Assign each user to a "day" to simulate gradual enrollment
rng = np.random.default_rng(42)
n_experiment_days = 28
df["day"] = rng.integers(1, n_experiment_days + 1, size=len(df))
df["group_label"] = df["treatment"].map({0: "control", 1: "treatment"})

# Run sequential test using the project utility
seq_results = sequential_test(
    df, metric_col="post_violation_rate", group_col="group_label",
    alpha=0.05, spending_func="obrien_fleming"
)

seq_df = pd.DataFrame(seq_results)
print(f"=== Sequential Testing: O'Brien-Fleming Spending ===")
print(f"Number of interim looks: {len(seq_df)}")
print(f"\n{'Day':>4} {'Info Frac':>10} {'Alpha Spent':>12} {'|Z|':>8} {'p-value':>10} {'Reject':>8}")
print("-" * 56)
for _, row in seq_df.iterrows():
    print(f"{row['day']:>4} {row['info_fraction']:>10.3f} {row['alpha_threshold']:>12.6f} "
          f"{abs(row['z_stat']):>8.3f} {row['p_value']:>10.6f} {str(row['reject']):>8}")

# Find first rejection day
rejections = seq_df[seq_df["reject"]]
if len(rejections) > 0:
    first_reject = rejections.iloc[0]
    print(f"\nFirst rejection at day {int(first_reject['day'])} "
          f"(info fraction = {first_reject['info_fraction']:.2f})")
    print(f"Could have stopped {n_experiment_days - int(first_reject['day'])} days early.")
else:
    print(f"\nNo early stopping: would need to run the full {n_experiment_days} days.")

# --- mSPRT (mixture Sequential Probability Ratio Test) ---
# Always-valid p-values: compute the likelihood ratio at each look
print("\n=== mSPRT: Always-Valid P-values ===")

tau_sq = 0.001  # mixing variance for the normal prior on the treatment effect
days_sorted = sorted(df["day"].unique())
msprt_results = []

for day in days_sorted:
    subset = df[df["day"] <= day]
    c = subset[subset["treatment"] == 0]["post_violation_rate"].values
    t = subset[subset["treatment"] == 1]["post_violation_rate"].values
    if len(c) < 50 or len(t) < 50:
        continue

    n_c, n_t = len(c), len(t)
    delta_hat = np.mean(t) - np.mean(c)
    sigma_sq = np.var(c, ddof=1) / n_c + np.var(t, ddof=1) / n_t

    # mSPRT likelihood ratio with normal mixing distribution
    V = sigma_sq  # variance of the effect estimator
    lambda_mix = np.sqrt(V / (V + tau_sq)) * np.exp(
        tau_sq * delta_hat**2 / (2 * V * (V + tau_sq))
    )
    # Always-valid p-value: 1 / max(1, lambda_mix)
    p_always_valid = min(1.0, 1.0 / max(lambda_mix, 1e-300))

    msprt_results.append({
        "day": day,
        "n_total": n_c + n_t,
        "delta_hat": delta_hat,
        "lambda_mix": lambda_mix,
        "p_always_valid": p_always_valid,
        "reject": p_always_valid < 0.05,
    })

msprt_df = pd.DataFrame(msprt_results)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: O'Brien-Fleming sequential monitoring
z_abs = seq_df["z_stat"].abs().values
boundaries = [abs(stats.norm.ppf(r / 2)) for r in seq_df["alpha_threshold"]]

axes[0].plot(seq_df["day"], z_abs, "o-", color=COLORS["treatment"],
             linewidth=2, markersize=4, label="|Z-statistic|")
axes[0].plot(seq_df["day"], boundaries, "--", color=COLORS["danger"],
             linewidth=2, label="OBF rejection boundary")

for _, r in seq_df[seq_df["reject"]].iterrows():
    axes[0].axvline(r["day"], color=COLORS["warning"], alpha=0.3, linewidth=3)

axes[0].set_xlabel("Day")
axes[0].set_ylabel("|Z-statistic|")
axes[0].set_title("O'Brien-Fleming Sequential Monitoring")
axes[0].legend(fontsize=9)

# Right: mSPRT always-valid p-values
axes[1].semilogy(msprt_df["day"], msprt_df["p_always_valid"], "o-",
                  color=COLORS["control"], linewidth=2, markersize=4,
                  label="Always-valid p-value")
axes[1].axhline(0.05, color=COLORS["danger"], linestyle="--", linewidth=1.5,
                label="alpha = 0.05")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Always-Valid P-value (log scale)")
axes[1].set_title("mSPRT Sequential Analysis")
axes[1].legend(fontsize=9)
axes[1].set_ylim(1e-10, 2)

plt.tight_layout()
plt.show()

# Summary
msprt_rejections = msprt_df[msprt_df["reject"]]
if len(msprt_rejections) > 0:
    first_msprt = msprt_rejections.iloc[0]
    print(f"\nmSPRT first rejection at day {int(first_msprt['day'])} "
          f"(n = {int(first_msprt['n_total']):,}, p_av = {first_msprt['p_always_valid']:.6f})")
else:
    print("\nmSPRT: No rejection during the experiment window.")

### Sequential Testing Takeaway

Sequential methods allow us to monitor the experiment continuously without inflating the false positive rate. The O'Brien-Fleming spending function is conservative early on (requiring very strong evidence to reject) and becomes more permissive as the experiment progresses. The mSPRT provides always-valid p-values that are valid at any stopping time.

In [ ]:
# ============================================================
# Subgroup Analysis
# ============================================================

# --- By Region ---
print("=== Treatment Effect by Region ===")
subgroup_results = []

for region in sorted(df["region"].unique()):
    region_df = df[df["region"] == region]
    c = region_df[region_df["treatment"] == 0]["post_violation_rate"].values
    t = region_df[region_df["treatment"] == 1]["post_violation_rate"].values
    res = two_sample_z_test(c, t)
    rel_effect = res["effect_size"] / res["control_mean"] if res["control_mean"] > 0 else 0
    subgroup_results.append({
        "subgroup": f"Region: {region}",
        "n_control": len(c),
        "n_treatment": len(t),
        "control_mean": res["control_mean"],
        "treatment_mean": res["treatment_mean"],
        "effect": res["effect_size"],
        "relative_effect": rel_effect,
        "ci_lower": res["ci_lower"],
        "ci_upper": res["ci_upper"],
        "p_value": res["p_value"],
    })

# --- By Content Type ---
print("\n=== Treatment Effect by Content Type ===")
for ctype in sorted(df["content_type_primary"].unique()):
    ctype_df = df[df["content_type_primary"] == ctype]
    c = ctype_df[ctype_df["treatment"] == 0]["post_violation_rate"].values
    t = ctype_df[ctype_df["treatment"] == 1]["post_violation_rate"].values
    res = two_sample_z_test(c, t)
    rel_effect = res["effect_size"] / res["control_mean"] if res["control_mean"] > 0 else 0
    subgroup_results.append({
        "subgroup": f"Content: {ctype}",
        "n_control": len(c),
        "n_treatment": len(t),
        "control_mean": res["control_mean"],
        "treatment_mean": res["treatment_mean"],
        "effect": res["effect_size"],
        "relative_effect": rel_effect,
        "ci_lower": res["ci_lower"],
        "ci_upper": res["ci_upper"],
        "p_value": res["p_value"],
    })

# --- By Account Age Quartile ---
print("\n=== Treatment Effect by Account Age Quartile ===")
df["age_quartile"] = pd.qcut(df["account_age_days"], q=4, labels=["Q1 (newest)", "Q2", "Q3", "Q4 (oldest)"])

for q in ["Q1 (newest)", "Q2", "Q3", "Q4 (oldest)"]:
    q_df = df[df["age_quartile"] == q]
    c = q_df[q_df["treatment"] == 0]["post_violation_rate"].values
    t = q_df[q_df["treatment"] == 1]["post_violation_rate"].values
    res = two_sample_z_test(c, t)
    rel_effect = res["effect_size"] / res["control_mean"] if res["control_mean"] > 0 else 0
    subgroup_results.append({
        "subgroup": f"Age: {q}",
        "n_control": len(c),
        "n_treatment": len(t),
        "control_mean": res["control_mean"],
        "treatment_mean": res["treatment_mean"],
        "effect": res["effect_size"],
        "relative_effect": rel_effect,
        "ci_lower": res["ci_lower"],
        "ci_upper": res["ci_upper"],
        "p_value": res["p_value"],
    })

sub_df = pd.DataFrame(subgroup_results)

# Print summary table
print("\n=== Subgroup Effects Summary ===")
print(f"{'Subgroup':<22} {'n (C/T)':>14} {'Ctrl Mean':>10} {'Effect':>10} {'Rel %':>8} {'95% CI':>24} {'p':>8}")
print("-" * 100)
for _, row in sub_df.iterrows():
    print(f"{row['subgroup']:<22} {row['n_control']:>6}/{row['n_treatment']:<6} "
          f"{row['control_mean']:>10.5f} {row['effect']:>10.5f} {row['relative_effect']:>7.1%} "
          f"[{row['ci_lower']:>10.5f}, {row['ci_upper']:>10.5f}] {row['p_value']:>8.4f}")

# --- Forest Plot ---
fig, ax = plt.subplots(figsize=(12, 8))

y_pos = np.arange(len(sub_df))
colors_forest = []
for _, row in sub_df.iterrows():
    if row["p_value"] < 0.05:
        colors_forest.append(COLORS["accent"])
    else:
        colors_forest.append(COLORS["neutral"])

ax.errorbar(sub_df["effect"], y_pos,
            xerr=[sub_df["effect"] - sub_df["ci_lower"],
                  sub_df["ci_upper"] - sub_df["effect"]],
            fmt="none", ecolor="gray", capsize=4, linewidth=1.5)

ax.scatter(sub_df["effect"], y_pos, c=colors_forest, s=100, zorder=5, edgecolors="black", linewidth=0.5)

# Overall effect line
ax.axvline(naive_result["effect_size"], color=COLORS["treatment"], linestyle="--",
           linewidth=1.5, alpha=0.7, label=f"Overall effect = {naive_result['effect_size']:.5f}")
ax.axvline(0, color="gray", linestyle="-", linewidth=1)

ax.set_yticks(y_pos)
ax.set_yticklabels(sub_df["subgroup"])
ax.set_xlabel("Treatment Effect (Absolute Change in Violation Rate)")
ax.set_title("Subgroup Analysis: Forest Plot of Treatment Effects")
ax.legend(loc="lower left", fontsize=10)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Guardrail Metrics Check
# ============================================================

print("=" * 60)
print("GUARDRAIL METRICS EVALUATION")
print("=" * 60)

guardrail_results = []

# --- 1. Posting Completion Rate ---
ctrl_pcr = ctrl["posting_completion_rate"].values
treat_pcr = treat["posting_completion_rate"].values
pcr_test = two_sample_z_test(ctrl_pcr, treat_pcr)

print("\n1. Posting Completion Rate")
print(f"   Control mean:   {pcr_test['control_mean']:.4f}")
print(f"   Treatment mean: {pcr_test['treatment_mean']:.4f}")
print(f"   Difference:     {pcr_test['effect_size']:.4f}")
print(f"   Relative:       {pcr_test['effect_size'] / pcr_test['control_mean']:.2%}")
print(f"   p-value:        {pcr_test['p_value']:.6f}")
print(f"   95% CI:         [{pcr_test['ci_lower']:.4f}, {pcr_test['ci_upper']:.4f}]")

pcr_threshold = -0.05  # cannot degrade by more than 5pp
pcr_pass = pcr_test["ci_lower"] > pcr_threshold
print(f"   Threshold:      > {pcr_threshold}")
print(f"   Status:         {'PASS' if pcr_pass else 'FAIL'}")
guardrail_results.append({
    "metric": "Posting Completion Rate",
    "control": pcr_test["control_mean"],
    "treatment": pcr_test["treatment_mean"],
    "difference": pcr_test["effect_size"],
    "p_value": pcr_test["p_value"],
    "threshold": "> -0.05",
    "status": "PASS" if pcr_pass else "FAIL",
})

# --- 2. Creator Activity (days_active_post) ---
ctrl_da = ctrl["days_active_post"].values.astype(float)
treat_da = treat["days_active_post"].values.astype(float)
da_test = two_sample_z_test(ctrl_da, treat_da)

print("\n2. Creator Activity (Days Active in 28-day Window)")
print(f"   Control mean:   {da_test['control_mean']:.4f}")
print(f"   Treatment mean: {da_test['treatment_mean']:.4f}")
print(f"   Difference:     {da_test['effect_size']:.4f}")
print(f"   p-value:        {da_test['p_value']:.6f}")
print(f"   95% CI:         [{da_test['ci_lower']:.4f}, {da_test['ci_upper']:.4f}]")

da_threshold = -0.5  # cannot drop by more than 0.5 days
da_pass = da_test["ci_lower"] > da_threshold
print(f"   Threshold:      > {da_threshold}")
print(f"   Status:         {'PASS' if da_pass else 'FAIL'}")
guardrail_results.append({
    "metric": "Days Active (28d)",
    "control": da_test["control_mean"],
    "treatment": da_test["treatment_mean"],
    "difference": da_test["effect_size"],
    "p_value": da_test["p_value"],
    "threshold": "> -0.5",
    "status": "PASS" if da_pass else "FAIL",
})

# --- 3. Warning Dismissal Rate (treatment only) ---
treat_warned = treat[treat["warned_count"] > 0]
n_warned = len(treat_warned)
dismiss_rate = (treat_warned["warning_dismissed"].sum() /
                treat_warned["warned_count"].sum())
heed_rate = 1 - dismiss_rate

print(f"\n3. Warning Dismissal Rate (Treatment Only)")
print(f"   Users who received warnings:   {n_warned:,} ({n_warned / n_treat:.1%} of treatment)")
print(f"   Total warnings shown:          {treat_warned['warned_count'].sum():,}")
print(f"   Warnings dismissed (posted):   {treat_warned['warning_dismissed'].sum():,}")
print(f"   Dismissal rate:                {dismiss_rate:.2%}")
print(f"   Heed rate (abandoned post):    {heed_rate:.2%}")

guardrail_results.append({
    "metric": "Warning Dismissal Rate",
    "control": float("nan"),
    "treatment": dismiss_rate,
    "difference": float("nan"),
    "p_value": float("nan"),
    "threshold": "Diagnostic",
    "status": "INFO",
})

# --- Summary table ---
print("\n" + "=" * 60)
print("GUARDRAIL SUMMARY")
print("=" * 60)
gr_df = pd.DataFrame(guardrail_results)
print(gr_df[["metric", "control", "treatment", "difference", "p_value", "status"]].to_string(
    index=False, float_format="%.4f", na_rep="N/A"
))

all_guardrails_pass = pcr_pass and da_pass
print(f"\nAll guardrails pass: {'YES' if all_guardrails_pass else 'NO'}")

## Decision Recommendation

### Summary of Findings

**Primary metric (Violation Rate)**:
- The friction warning reduces the post-experiment violation rate. The CUPED-adjusted analysis confirms the effect with a narrower confidence interval.
- The effect is strongest in SEA and weakest in EU, consistent with regional differences in user response to warnings.

**Guardrail metrics**:
- **Posting completion rate** decreases for treatment users (expected, since some users heed the warning and abandon their post). The decrease stays within the acceptable threshold.
- **Creator activity** (days active) shows negligible difference between treatment and control.
- **Warning dismissal rate** indicates that roughly 60% of warned users dismiss the warning and post anyway, while ~40% heed the warning.

**Subgroup effects**:
- Treatment is effective across all regions, with the strongest signal in SEA.
- Effect is consistent across content types and account age quartiles.

### Cost-Benefit Analysis

In [ ]:
# ============================================================
# Cost-Benefit Calculation
# ============================================================

# Assumptions
total_monthly_posts = 500_000_000  # 500M posts/month platform-wide
cost_per_violation = 2.50  # moderation cost per violation (review + action)
cost_per_abandoned_post = 0.005  # lost engagement value per abandoned post

# Estimated effects (from CUPED analysis)
vr_control = naive_result["control_mean"]
vr_treatment = naive_result["treatment_mean"]
pcr_control = pcr_test["control_mean"]
pcr_treatment = pcr_test["treatment_mean"]

# Monthly violation reduction
violations_control = total_monthly_posts * vr_control
violations_treatment = total_monthly_posts * vr_treatment
violations_prevented = violations_control - violations_treatment

# Monthly posts abandoned
posts_abandoned = total_monthly_posts * (pcr_control - pcr_treatment)

# Net benefit
benefit = violations_prevented * cost_per_violation
cost = posts_abandoned * cost_per_abandoned_post
net_benefit = benefit - cost

print("=== Monthly Cost-Benefit Projection ===")
print(f"Total monthly posts:           {total_monthly_posts:>15,}")
print(f"Violations prevented:          {violations_prevented:>15,.0f}")
print(f"Posts abandoned:               {posts_abandoned:>15,.0f}")
print(f"")
print(f"Benefit (moderation savings):  ${benefit:>14,.0f}")
print(f"Cost (lost engagement):        ${cost:>14,.0f}")
print(f"Net monthly benefit:           ${net_benefit:>14,.0f}")
print(f"")
print(f"Benefit/Cost ratio:            {benefit / max(cost, 1):.1f}x")

print("\n" + "=" * 60)
print("PRODUCT RECOMMENDATION")
print("=" * 60)
print("""
The friction warning experiment demonstrates a statistically significant
reduction in violation rate with acceptable impact on posting completion
and creator activity. The CUPED-adjusted analysis confirms the treatment
effect is robust, and sequential testing shows the effect was detectable
well before the experiment concluded. All guardrail metrics pass their
pre-specified thresholds.

Recommendation: LAUNCH the friction warning globally. The intervention
produces meaningful safety improvements (fewer violations reaching users)
with minimal friction cost. For SEA markets, where the effect is strongest,
prioritize an accelerated rollout. Continue monitoring for novelty effects
and long-term habituation; if the violation reduction decays below 50% of
the observed effect after 90 days, revisit the warning design.
""")

In [ ]:
# ============================================================
# Post-Experiment Monitoring: Novelty Effect Detection
# ============================================================

# Simulate daily treatment effects over the 28-day experiment
# to check for novelty (inflated early effect) or temporal decay

daily_effects = []
for day in sorted(df["day"].unique()):
    day_data = df[df["day"] == day]
    c = day_data[day_data["treatment"] == 0]["post_violation_rate"].values
    t = day_data[day_data["treatment"] == 1]["post_violation_rate"].values
    if len(c) > 50 and len(t) > 50:
        effect = np.mean(t) - np.mean(c)
        se = np.sqrt(np.var(c, ddof=1) / len(c) + np.var(t, ddof=1) / len(t))
        daily_effects.append({
            "day": day,
            "effect": effect,
            "se": se,
            "n_control": len(c),
            "n_treatment": len(t),
        })

daily_df = pd.DataFrame(daily_effects)

# Fit separate trend lines for early vs late periods
early = daily_df[daily_df["day"] <= 7]
late = daily_df[daily_df["day"] > 7]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Daily treatment effects with trend
axes[0].plot(daily_df["day"], daily_df["effect"], "o-",
             color=COLORS["treatment"], linewidth=2, markersize=5, alpha=0.7)
axes[0].fill_between(daily_df["day"],
                     daily_df["effect"] - 1.96 * daily_df["se"],
                     daily_df["effect"] + 1.96 * daily_df["se"],
                     alpha=0.15, color=COLORS["treatment"])
axes[0].axhline(0, color="gray", linestyle="-", linewidth=1)
axes[0].axhline(naive_result["effect_size"], color=COLORS["control"],
                linestyle="--", alpha=0.5, label=f"Overall effect = {naive_result['effect_size']:.5f}")

# Highlight novelty window
axes[0].axvspan(1, 7, alpha=0.08, color=COLORS["warning"], label="Novelty window (days 1-7)")

# Fit overall trend
if len(daily_df) > 3:
    z_all = np.polyfit(daily_df["day"], daily_df["effect"], 1)
    axes[0].plot(daily_df["day"], np.polyval(z_all, daily_df["day"]),
                "--", color=COLORS["danger"], linewidth=2,
                label=f"Trend slope = {z_all[0]:.6f}/day")

axes[0].set_xlabel("Experiment Day")
axes[0].set_ylabel("Treatment Effect (VR difference)")
axes[0].set_title("Daily Treatment Effect Over Time")
axes[0].legend(fontsize=8)

# Right: Early vs late cohort comparison
early_effect = early["effect"].mean()
late_effect = late["effect"].mean()
early_se = np.sqrt((early["se"]**2).mean() / len(early))
late_se = np.sqrt((late["se"]**2).mean() / len(late))

cohort_labels = ["Early (days 1-7)", "Late (days 8-28)"]
cohort_effects = [early_effect, late_effect]
cohort_ses = [early_se, late_se]
cohort_colors = [COLORS["warning"], COLORS["accent"]]

axes[1].barh([0, 1], cohort_effects, color=cohort_colors, alpha=0.7, height=0.4)
axes[1].errorbar(cohort_effects, [0, 1],
                 xerr=[1.96 * s for s in cohort_ses],
                 fmt="o", color="black", capsize=5, markersize=8)
axes[1].axvline(0, color="gray", linestyle="-", linewidth=1)
axes[1].set_yticks([0, 1])
axes[1].set_yticklabels(cohort_labels)
axes[1].set_xlabel("Treatment Effect")
axes[1].set_title("Novelty Check: Early vs Late Cohorts")

for i, (eff, label) in enumerate(zip(cohort_effects, cohort_labels)):
    axes[1].text(eff + 0.0001, i, f"{eff:.5f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

# Statistical test for novelty effect
pooled_se_novelty = np.sqrt(early_se**2 + late_se**2)
z_novelty = (early_effect - late_effect) / pooled_se_novelty if pooled_se_novelty > 0 else 0
p_novelty = 2 * (1 - stats.norm.cdf(abs(z_novelty)))

print(f"\n=== Novelty Effect Test ===")
print(f"Early period effect (days 1-7):  {early_effect:.6f}")
print(f"Late period effect (days 8-28):  {late_effect:.6f}")
print(f"Difference:                      {early_effect - late_effect:.6f}")
print(f"Z-statistic:                     {z_novelty:.4f}")
print(f"p-value:                         {p_novelty:.4f}")
print(f"Novelty effect {'detected' if p_novelty < 0.05 else 'NOT detected'} at alpha = 0.05")

# Long-run projection
if len(daily_df) > 3:
    slope = z_all[0]
    intercept = z_all[1]
    effect_day_90 = slope * 90 + intercept
    effect_day_180 = slope * 180 + intercept
    print(f"\n=== Long-Run Projection (Linear Extrapolation) ===")
    print(f"Projected effect at day 90:  {effect_day_90:.6f}")
    print(f"Projected effect at day 180: {effect_day_180:.6f}")
    print(f"Current (day {int(daily_df['day'].max())}):          {naive_result['effect_size']:.6f}")
    if abs(effect_day_90) < abs(naive_result['effect_size']) * 0.5:
        print("WARNING: Effect may decay below 50% of observed by day 90.")
    else:
        print("Effect appears stable -- no significant temporal decay projected.")

### Post-Experiment Monitoring Takeaway

Novelty effects are a common threat in friction interventions: users may initially comply with warnings (high heed rate) but gradually learn to dismiss them habitually. The daily treatment effect analysis above checks for this pattern. If the effect decays significantly over time, the intervention's long-term value is lower than the experiment suggests, and the team should consider refreshing the warning design or targeting only high-risk posts.

## Summary and Key Takeaways

### What We Learned About Friction Warnings

1. **Friction warnings reduce violation rates.** The interstitial warning successfully reduces the post-experiment violation rate, with the CUPED-adjusted analysis confirming the effect with higher precision.

2. **The cost is manageable.** Posting completion rate decreases slightly (users who heed the warning abandon their post), but the decrease stays within the pre-specified guardrail threshold. Creator activity (days active) is essentially unaffected.

3. **Regional heterogeneity matters.** SEA shows the strongest response to the warning, while EU shows the weakest. This has implications for prioritizing rollout and may reflect cultural differences in response to authority cues.

4. **About 60% of warnings are dismissed.** This means the violation reduction comes from both (a) users who heed the warning and abandon violating posts, and (b) a behavioral shift among users who dismiss the warning but still moderate their content.

### Methodological Takeaways

| Method | Insight |
|--------|--------|
| **CUPED** | Pre-experiment violation rate is a strong covariate, yielding substantial variance reduction and narrower CIs |
| **Sequential testing** | O'Brien-Fleming and mSPRT both allow valid early stopping, potentially saving days of experimentation |
| **Subgroup analysis** | Forest plots reveal heterogeneity across regions, content types, and account age quartiles |
| **Guardrail monitoring** | Separate thresholds for posting completion and creator activity ensure no unacceptable side effects |
| **Novelty detection** | Comparing early vs late cohorts and fitting time trends catches inflated initial effects |

### Connection to Trust & Safety DS Interviews

This notebook demonstrates the end-to-end workflow a Data Scientist on a Trust & Safety team would execute:

- **Metric definition**: Choosing an OEC and guardrails specific to content safety
- **Power analysis**: Accounting for low base rates typical of violation metrics
- **Variance reduction**: Using CUPED with pre-experiment data to get more precise estimates
- **Sequential monitoring**: Enabling responsible early stopping without inflating false positives
- **Heterogeneity**: Identifying which user segments benefit most (or are harmed)
- **Decision framework**: Translating statistical results into a launch/no-launch recommendation with cost-benefit analysis
- **Post-launch monitoring**: Watching for novelty effects and long-term decay